# Módulo 03 · Aula 1: RAG como grafo

**Duração:** 45 min

## Objetivos de aprendizagem

- Transformar o RAG do módulo 02 (uma função: buscar e gerar) em um grafo LangGraph com nós `retrieve` e `generate`.
- Desenhar o **estado de um RAG**: quais chaves existem, quem escreve cada uma e o que o front-end precisa ler.
- Manter uma conversa com um checkpointer (`InMemorySaver`) e entender por que a **pergunta de continuação** busca mal.
- Resolver a continuação com o nó `condense` e o `CONDENSE_PROMPT`, comparando a busca antes e depois.
- Separar **chaves do turno** (zeradas a cada pergunta) de **chaves da conversa** (acumuladas por reducers).

## Pré-requisitos

- Módulo 02: `HybridRetriever` e `generate_answer` (resposta com citações).
- Curso de LangGraph: `StateGraph`, reducers (`add_messages`), checkpointers e `thread_id`. Não vamos reensinar esses
  assuntos: aqui eles são ferramentas para resolver problemas do RAG.

## O que vamos construir

Três versões do mesmo assistente, cada uma resolvendo um problema da anterior: primeiro o RAG vira grafo; depois ganha
memória de conversa; por fim ganha o nó `condense`, que transforma "E ele aguenta chuva?" em uma pergunta completa
antes da busca. No fim, conhecemos a seção **Estado** do produto em
[src/graph/corrective_rag.py](../src/graph/corrective_rag.py).

```mermaid
flowchart LR
    S([START]) --> C[condense<br/>LLM, só se houver histórico] --> R[retrieve<br/>busca híbrida] --> G[generate<br/>LLM com citações] --> E([END])
    CP[(checkpointer<br/>messages e usage)] -.-> C
```

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. Do RAG em uma função ao RAG como grafo

No módulo 02, o RAG inteiro cabia em duas linhas: buscar os trechos e gerar a resposta com citações. Vamos carregar o
índice principal e relembrar essa versão.

In [2]:
from src.rag.generation import generate_answer
from src.rag.retrieval import HybridRetriever
from src.rag.vectorstore import index_exists, load_index

if not index_exists():
    from src.rag.ingestion import index_folder

    index_folder()

retriever = HybridRetriever(index=load_index())
print(f"Retriever pronto: busca {retriever.mode}, {retriever.k} trechos por pergunta")


def origin(chunk) -> str:
    """Origem curta de um trecho: arquivo e, nos PDFs, a página."""
    md = chunk.metadata
    return md["source"] + (f" p.{md['page']}" if md.get("page") else "")


def answer_question(question: str) -> dict:
    chunks = retriever.invoke(question)
    return generate_answer(question, chunks)


result = answer_question("Em quantos dias posso trocar um produto?")
print(result["answer"])
print("Fontes:", [s["source"] for s in result["sources"]])

2026-09-27 22:32:23 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


Retriever pronto: busca hybrid, 4 trechos por pergunta


2026-09-27 22:32:24 | INFO     | src.models | Inicializando LLM | provedor=azure | deployment=gpt-5.4-mini


Você pode trocar um produto em até 30 dias corridos após a entrega. O produto deve estar sem sinais de uso, com todos os acessórios e na embalagem original. O frete da troca é gratuito. [1]
Fontes: ['faq.csv']


Funciona, e para uma pergunta isolada e bem escrita é tudo de que precisamos. O problema aparece no uso real. As três
primeiras aulas deste módulo resolvem três falhas desse RAG "em linha reta":

| Falha | Exemplo | Solução | Aula |
|---|---|---|---|
| Pergunta de continuação | "E ele aguenta chuva?" | Conversa com checkpointer e nó `condense` | 3.1 (esta) |
| Busca que falha | "A TechNova está contratando?" | Avaliar a relevância, reescrever a consulta, desistir com segurança | 3.2 |
| Documentos **e** dados | "Meu pedido PED-1001 ainda está na garantia?" | Agente com a busca e a consulta de pedidos como ferramentas | 3.3 |

Todas as soluções precisam de algo que uma função simples não tem: **estado** que atravessa várias etapas (e vários
turnos), **decisões** no meio do caminho e **ciclos** com limite. É exatamente o que o LangGraph oferece.

## 2. O estado de um RAG

Antes de escrever nós, desenhamos o estado. A pergunta que guia o desenho é: **o que cada etapa precisa ler e o que ela
produz?**

| Chave | Quem escreve | Quem lê | Por que fica no estado |
|---|---|---|---|
| `question` | quem chama o grafo | `retrieve`, `generate` | é a entrada |
| `chunks` | `retrieve` | `generate` | o front-end mostra as fontes; nós futuros (como a avaliação) também leem |
| `answer` | `generate` | quem chama | é a saída |
| `sources` | `generate` | front-end | arquivo, título e página de cada citação |

Guardamos os trechos como `Document`, e não só como texto: os metadados (`source`, `page`, `section`) são o que vira
citação.

Como vimos no curso de LangGraph, `stream_mode="updates"` mostra o que **cada nó escreveu** no estado: dá para
ver o que a busca trouxe antes de ver a resposta.

In [3]:
from typing import TypedDict

from langchain_core.documents import Document
from langgraph.graph import END, START, StateGraph


class SimpleRAGState(TypedDict, total=False):
    question: str  # entrada: a pergunta do cliente
    chunks: list[Document]  # escrita por retrieve
    answer: str  # escrita por generate
    sources: list[dict]  # escrita por generate


def retrieve(state: SimpleRAGState) -> dict:
    return {"chunks": retriever.invoke(state["question"])}


def generate(state: SimpleRAGState) -> dict:
    result = generate_answer(state["question"], state["chunks"])
    return {"answer": result["answer"], "sources": result["sources"]}


builder = StateGraph(SimpleRAGState)
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
simple_graph = builder.compile()


def summarize(values: dict) -> str:
    """Resumo legível de uma atualização de estado (trechos viram origens; textos longos são cortados)."""
    parts = []
    for key, value in (values or {}).items():
        if key == "chunks":
            value = [origin(c) for c in value]
        elif key == "sources":
            value = [s["source"] for s in value]
        elif key == "messages":
            value = [m.content[:80] for m in value]
        elif isinstance(value, str) and len(value) > 100:
            value = value[:100] + "..."
        parts.append(f"{key}={value}")
    return "; ".join(parts)


for step in simple_graph.stream({"question": "Quanto tempo dura a bateria do Fone Nova Buds?"}, stream_mode="updates"):
    for node, values in step.items():
        print(f"[{node}] {summarize(values)}\n")

[retrieve] chunks=['faq.csv', 'manuals/fn-001_nova_buds.pdf p.4', 'manuals/fn-001_nova_buds.pdf p.3', 'manuals/fn-001_nova_buds.pdf p.5']



[generate] answer=A bateria do Fone Nova Buds dura até 6 horas com o cancelamento de ruído ligado e até 8 horas com o ...; sources=['faq.csv', 'manuals/fn-001_nova_buds.pdf']



**Observe:** o grafo faz exatamente o que a função fazia, em dois nós. O ganho ainda não é funcional: é de
**visibilidade** (cada etapa aparece no stream) e de **extensibilidade** (daqui em diante, cada melhoria é um nó ou uma
aresta nova, sem reescrever o resto).

## 3. Conversa com checkpointer

Um assistente de atendimento conversa. Para isso trocamos a entrada `question` por `messages`, com o reducer
`add_messages`, e compilamos com um checkpointer: cada `thread_id` guarda uma conversa. Por enquanto, a pergunta do
turno é simplesmente a última mensagem do cliente, **do jeito que ele escreveu**. O `generate` recebe o
histórico (as últimas mensagens) e grava a resposta como `AIMessage`.

In [4]:
from typing import Annotated

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages

HISTORY_MESSAGES = 6  # recorte da conversa enviado ao modelo


class ChatRAGState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]  # a conversa inteira
    question: str  # a pergunta usada na busca, neste turno
    chunks: list[Document]
    sources: list[dict]


def read_question(state: ChatRAGState) -> dict:
    # Sem condensar: a pergunta do turno é a última mensagem, exatamente como o cliente escreveu.
    return {"question": state["messages"][-1].content}


def retrieve_chat(state: ChatRAGState) -> dict:
    return {"chunks": retriever.invoke(state["question"])}


def generate_chat(state: ChatRAGState) -> dict:
    history = state["messages"][:-1][-HISTORY_MESSAGES:]
    result = generate_answer(state["question"], state["chunks"], history)
    return {"messages": [AIMessage(result["answer"])], "sources": result["sources"]}


def build_chat_graph(first_node, first_name: str):
    builder = StateGraph(ChatRAGState)
    builder.add_node(first_name, first_node)
    builder.add_node("retrieve", retrieve_chat)
    builder.add_node("generate", generate_chat)
    builder.add_edge(START, first_name)
    builder.add_edge(first_name, "retrieve")
    builder.add_edge("retrieve", "generate")
    builder.add_edge("generate", END)
    return builder.compile(checkpointer=InMemorySaver())


naive_chat = build_chat_graph(read_question, "read_question")


def chat(graph, text: str, thread_id: str) -> dict:
    """Um turno da conversa: mostra a pergunta usada na busca, os trechos e a resposta."""
    state = graph.invoke({"messages": [HumanMessage(text)]}, {"configurable": {"thread_id": thread_id}})
    print(f"Cliente: {text}")
    print(f"  busca por: {state['question']}")
    print(f"  trechos:   {[origin(c) for c in state['chunks']]}")
    print(f"  Assistente: {state['messages'][-1].content}\n")
    return state


naive_turns = [
    chat(naive_chat, "Quanto tempo dura a bateria do Fone Nova Buds?", "naive"),
    chat(naive_chat, "E ele aguenta chuva?", "naive"),
]

Cliente: Quanto tempo dura a bateria do Fone Nova Buds?
  busca por: Quanto tempo dura a bateria do Fone Nova Buds?
  trechos:   ['faq.csv', 'manuals/fn-001_nova_buds.pdf p.4', 'manuals/fn-001_nova_buds.pdf p.3', 'manuals/fn-001_nova_buds.pdf p.5']
  Assistente: A bateria do Fone Nova Buds dura até 6 horas com o cancelamento de ruído ligado e até 8 horas com o cancelamento de ruído desligado. Com as recargas do estojo, a autonomia total chega a 24 horas com ANC ligado. [1][3]



Cliente: E ele aguenta chuva?
  busca por: E ele aguenta chuva?
  trechos:   ['help_center/nova_sport_reviews.html', 'faq.csv', 'faq.csv', 'manuals/sm-004_nova_mini.pdf p.2']
  Assistente: Não encontrei essa informação nos documentos da TechNova. Se quiser, posso encaminhar sua dúvida para um atendente.



**Observe:** o checkpointer fez a parte dele (o histórico está lá e chegou ao `generate`), mas o segundo turno **falha**.
A busca trouxe as avaliações do Fone Nova Sport e trechos de outros aparelhos, nenhum do manual do Nova Buds. Com
isso, o modelo ora responde "sim" sobre o Fone Nova Sport (errado com cara de certo), ora diz que não encontrou,
embora a base saiba a resposta: o Fone Nova Buds **não** tem certificação de resistência à água. O problema foi a
**busca**: "E ele aguenta chuva?" não diz de qual produto estamos falando.

> ⚠️ Memória de conversa no `generate` **não** conserta a busca. O retriever não lê o histórico: ele só vê a string
> que recebe.

## 4. Por que a continuação busca mal

Vamos isolar o problema sem chamar o LLM: buscar a pergunta de continuação sozinha e a mesma pergunta escrita de forma
completa (por enquanto, à mão).

In [5]:
pairs = [
    ("E ele aguenta chuva?", "O Fone Nova Buds aguenta chuva?"),
    ("E se ele veio com defeito?", "Em quantos dias posso trocar um produto que veio com defeito?"),
]
for follow_up, complete in pairs:
    for label, query in (("sozinha ", follow_up), ("completa", complete)):
        print(f"{label} | {query}")
        for chunk in retriever.search(query, k=4):
            print(f"           {origin(chunk)}")
    print()

sozinha  | E ele aguenta chuva?


           help_center/nova_sport_reviews.html
           faq.csv
           faq.csv
           manuals/sm-004_nova_mini.pdf p.2
completa | O Fone Nova Buds aguenta chuva?


           manuals/fn-001_nova_buds.pdf p.5
           faq.csv
           help_center/nova_sport_reviews.html
           help_center/warranty_claim_step_by_step.html

sozinha  | E se ele veio com defeito?


           policies/exchange.md
           help_center/damaged_product.html
           help_center/warranty_claim_step_by_step.html
           help_center/damaged_product.html
completa | Em quantos dias posso trocar um produto que veio com defeito?


           faq.csv
           help_center/exchange_for_another_model.html
           policies/exchange.md
           faq.csv



**Observe:**

- **"E ele aguenta chuva?"** falha feio: sem o nome do produto, a busca encontra **outros** aparelhos que lidam com
  água (Fone Nova Sport, Caixa de Som Nova Boom, smartphones). A versão completa traz o manual do Fone Nova Buds.
- **"E se ele veio com defeito?"**, a continuação de "Em quantos dias posso trocar um produto?",
  sobrevive por sorte: a palavra "defeito" é forte o bastante para achar a política de troca de produto com defeito. Mas
  a versão completa traz também o prazo de troca, que é o assunto da conversa.

Nem toda continuação quebra a busca, mas quando quebra, quebra **em silêncio**: o sistema responde "não encontrei" (ou,
pior, responde sobre o produto errado). A solução é reescrever a pergunta usando o histórico **antes** de buscar.

## 5. O nó `condense`

O `CONDENSE_PROMPT`, de [src/prompts.py](../src/prompts.py), recebe o histórico e devolve a última mensagem reescrita
como pergunta completa. Três decisões de desenho no nó:

1. **Primeira pergunta não chama o modelo**: sem histórico, não há referência a resolver.
2. **Recorte do histórico**: só as últimas `HISTORY_MESSAGES` mensagens, como no `generate`.
3. **Plano B**: se o modelo devolver texto vazio, seguimos com a pergunta original.

Para trocar o comportamento do grafo, basta trocar **um nó**: `read_question` sai, `condense` entra.

In [6]:
from langchain_core.output_parsers import StrOutputParser

from src.models import get_llm
from src.prompts import CONDENSE_PROMPT

print(CONDENSE_PROMPT.messages[0].prompt.template, "\n")


def condense(state: ChatRAGState) -> dict:
    messages = state["messages"]
    if len(messages) == 1:  # primeira pergunta: nada a resolver
        return {"question": messages[-1].content}
    chain = CONDENSE_PROMPT | get_llm() | StrOutputParser()
    question = chain.invoke({"history": messages[-HISTORY_MESSAGES:]}).strip()
    return {"question": question or messages[-1].content}


condense_chat = build_chat_graph(condense, "condense")

condensed_turns = [
    chat(condense_chat, "Quanto tempo dura a bateria do Fone Nova Buds?", "condense"),
    chat(condense_chat, "E ele aguenta chuva?", "condense"),
    chat(condense_chat, "E se ele veio com defeito?", "condense"),
]

Reescreva a última mensagem do cliente como uma pergunta completa e independente, usando o histórico só para resolver referências (ex.: 'e o dele?', 'quanto tempo?'). Não responda a pergunta. Devolva apenas a pergunta reescrita. 



Cliente: Quanto tempo dura a bateria do Fone Nova Buds?
  busca por: Quanto tempo dura a bateria do Fone Nova Buds?
  trechos:   ['faq.csv', 'manuals/fn-001_nova_buds.pdf p.4', 'manuals/fn-001_nova_buds.pdf p.3', 'manuals/fn-001_nova_buds.pdf p.5']
  Assistente: A bateria do Fone Nova Buds dura até 6 horas com o cancelamento de ruído ligado e até 8 horas com o cancelamento de ruído desligado. Com as recargas do estojo, a autonomia total chega a 24 horas com ANC ligado. [1][3]



Cliente: E ele aguenta chuva?
  busca por: O Fone Nova Buds é resistente à água?
  trechos:   ['help_center/warranty_claim_step_by_step.html', 'help_center/nova_sport_reviews.html', 'policies/warranty.md', 'faq.csv']
  Assistente: Não. O Fone Nova Buds não tem certificação de resistência à água, e danos por líquidos não são cobertos pela garantia. [3]



Cliente: E se ele veio com defeito?
  busca por: Se o Fone Nova Buds veio com defeito, você pode acionar a garantia para solicitar reparo ou substituição?
  trechos:   ['manuals/fn-001_nova_buds.pdf p.5', 'manuals/fn-001_nova_buds.pdf p.5', 'help_center/warranty_claim_step_by_step.html', 'help_center/warranty_claim_step_by_step.html']
  Assistente: Sim. Para acionar a garantia do Nova Buds, é obrigatório informar o número do pedido (formato PED-1234), além da descrição do problema, fotos ou vídeo e o número de série. Você pode fazer isso em Minha Conta > Meus Pedidos > "Acionar garantia" ou pelos canais de atendimento. Depois, você recebe um código de postagem gratuito dos Correios para enviar o produto à assistência técnica credenciada. Se a assistência não resolver em até 30 dias corridos após receber o produto, você pode escolher troca por um produto igual ou equivalente, abatimento proporcional do valor pago ou reembolso, sujeito às regras normais e aprovação de um atendente. [1][2

**Observe** (compare o segundo turno com o da seção 3):

- O `condense` resolveu a referência ("ele" quem?) usando o histórico. A busca passou a trazer trechos sobre o Fone
  Nova Buds (nesta execução, a política de garantia; em outras, o manual) e a resposta ficou correta e com fonte: ele
  não tem certificação de resistência à água.
- No terceiro turno, "E se ele veio com defeito?" virou uma pergunta sobre o **Fone Nova Buds** com defeito: a
  referência "ele" foi resolvida para o produto da conversa, não para o assunto troca.
- O custo: **uma chamada a mais** ao modelo por turno, a partir do segundo. Na aula 3.2 vamos medir esse custo junto com
  os outros nós.

> 💡 **Dica:** o `condense` também ajuda quando a pergunta é autossuficiente, porque padroniza a consulta (corrige
> "vcs", completa frases soltas). O preço é que ele pode **distorcer** a pergunta. Por isso guardamos `question` no
> estado: no stream e nos logs dá para ver exatamente o que foi buscado.

## 6. Chaves do turno e chaves da conversa

### 6.1 O que o checkpointer guarda

Com um checkpointer, **o estado inteiro** é persistido entre as perguntas, não só as mensagens:

In [7]:
snapshot = condense_chat.get_state({"configurable": {"thread_id": "condense"}})
values = snapshot.values
print(f"messages: {len(values['messages'])} mensagens (a conversa inteira)")
print(f"question: {values['question']}")
print(f"chunks:   {len(values['chunks'])} trechos (só os do último turno)")
print(f"sources:  {[s['source'] for s in values['sources']]}")

messages: 6 mensagens (a conversa inteira)
question: Se o Fone Nova Buds veio com defeito, você pode acionar a garantia para solicitar reparo ou substituição?
chunks:   4 trechos (só os do último turno)
sources:  ['manuals/fn-001_nova_buds.pdf', 'manuals/fn-001_nova_buds.pdf', 'help_center/warranty_claim_step_by_step.html']


Isso divide as chaves em dois grupos com regras diferentes:

| Grupo | Chaves (no produto) | Regra |
|---|---|---|
| **Conversa** | `messages`, `usage` | acumulam turno a turno, com **reducers** (`add_messages`, `accumulate_usage`) |
| **Turno** | `question`, `queries`, `chunks`, `relevant`, `rewrites`, `sources`, `found` | valem só para a pergunta atual e precisam ser **zeradas** no início de cada turno |

Neste grafo simples, todo turno sobrescreve `question`, `chunks` e `sources`, então nada "vaza". Mas o RAG corretivo da
próxima aula tem um **contador** de reescritas e caminhos que não escrevem todas as chaves. Veja o que acontece com um
contador que não é zerado. O grafo abaixo não chama nenhum modelo: a "busca" sempre falha, e o grafo reescreve até
duas vezes antes de desistir.

In [8]:
from typing import Literal

MAX_ATTEMPTS = 2


class CounterState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]
    rewrites: int  # chave do turno: quantas reescritas NESTA pergunta


def build_counter_graph(reset: bool):
    def start_turn(state: CounterState) -> dict:
        return {"rewrites": 0} if reset else {}

    def search_stub(state: CounterState) -> dict:
        return {}  # busca que nunca encontra nada

    def route(state: CounterState) -> Literal["rewrite", "give_up"]:
        return "rewrite" if state.get("rewrites", 0) < MAX_ATTEMPTS else "give_up"

    def rewrite(state: CounterState) -> dict:
        return {"rewrites": state.get("rewrites", 0) + 1}

    def give_up(state: CounterState) -> dict:
        return {"messages": [AIMessage("Não encontrei.")]}

    builder = StateGraph(CounterState)
    for name, node in [("start_turn", start_turn), ("search", search_stub), ("rewrite", rewrite), ("give_up", give_up)]:
        builder.add_node(name, node)
    builder.add_edge(START, "start_turn")
    builder.add_edge("start_turn", "search")
    builder.add_conditional_edges("search", route, ["rewrite", "give_up"])
    builder.add_edge("rewrite", "search")
    builder.add_edge("give_up", END)
    return builder.compile(checkpointer=InMemorySaver())


for reset in (False, True):
    graph = build_counter_graph(reset)
    config = {"configurable": {"thread_id": "counter"}}
    print("zerando no início do turno" if reset else "SEM zerar")
    for turn in (1, 2, 3):
        nodes = [node for step in graph.stream({"messages": [HumanMessage("vagas?")]}, config, stream_mode="updates") for node in step]
        print(f"  turno {turn}: {nodes.count('rewrite')} reescritas | caminho: {' -> '.join(nodes)}")
    print()

SEM zerar
  turno 1: 2 reescritas | caminho: start_turn -> search -> rewrite -> search -> rewrite -> search -> give_up
  turno 2: 0 reescritas | caminho: start_turn -> search -> give_up
  turno 3: 0 reescritas | caminho: start_turn -> search -> give_up

zerando no início do turno
  turno 1: 2 reescritas | caminho: start_turn -> search -> rewrite -> search -> rewrite -> search -> give_up
  turno 2: 2 reescritas | caminho: start_turn -> search -> rewrite -> search -> rewrite -> search -> give_up
  turno 3: 2 reescritas | caminho: start_turn -> search -> rewrite -> search -> rewrite -> search -> give_up



**Observe:** sem zerar, o turno 2 começa com `rewrites=2`, herdado do turno 1, e desiste **sem tentar nada**. Nenhum
erro, nenhum aviso: só um assistente que fica pior a partir da segunda pergunta. É o tipo de bug que passa em todos os
testes de uma pergunta só.

A regra do produto: **o primeiro nó do turno devolve todas as chaves do turno com valores iniciais**. No RAG corretivo,
esse nó é o próprio `condense`:

```python
# src/graph/corrective_rag.py, final do nó condense
return {
    "question": question,
    "queries": [question],
    "chunks": [],
    "relevant": [],
    "rewrites": 0,
    "sources": [],
    "found": False,
    "usage": usage,
}
```

### 6.2 Chaves da conversa: reducers

As chaves da conversa nunca são zeradas; cada nó **acrescenta** a elas, e quem decide como acrescentar é o reducer:

- `messages` usa `add_messages`: anexa a pergunta (enviada por quem chama o grafo) e a resposta (escrita pelo `generate`
  ou pelo `no_answer`).
- `usage` usa `accumulate_usage`: cada nó que chama o modelo devolve o consumo **daquela chamada**, e o reducer soma ao
  total da conversa. Nenhum nó precisa ler o total antes de escrever, e dois nós nunca se atropelam.

> 💡 **Dica:** repare que o `condense` do produto devolve `"usage": usage`, que pode ser `{}` no primeiro turno. Com um
> reducer de soma, devolver um consumo vazio é inofensivo; sem reducer, **apagaria** o total da conversa.

## 7. Do notebook para o projeto: o estado em `src/graph/corrective_rag.py`

A seção **Estado** de [src/graph/corrective_rag.py](../src/graph/corrective_rag.py) reúne o que desenhamos nesta aula:

- `RAGState`: o contrato entre os nós, com as chaves da conversa (com reducers) e as do turno;
- `accumulate_usage`: o reducer de consumo de tokens (usa `add_usage`, de [src/models.py](../src/models.py));
- `RelevanceGrade`: o schema da avaliação de relevância, que o nó `grade` usa na próxima aula.

Vamos inspecionar o estado sem ler o arquivo: `get_type_hints(..., include_extras=True)` mostra o tipo e o reducer de
cada chave.

In [9]:
from typing import get_type_hints

from src.graph.corrective_rag import HISTORY_MESSAGES as PRODUCT_HISTORY
from src.graph.corrective_rag import RAGState, RelevanceGrade, accumulate_usage

hints = get_type_hints(RAGState, include_extras=True)
print(f"{'chave':<10} | {'escopo':<8} | reducer")
for key, hint in hints.items():
    reducer = getattr(hint, "__metadata__", (None,))[0]
    scope = "conversa" if reducer else "turno"
    print(f"{key:<10} | {scope:<8} | {reducer.__name__ if reducer else '(sobrescreve)'}")
print(f"\nRecorte do histórico no produto: HISTORY_MESSAGES = {PRODUCT_HISTORY}")

chave      | escopo   | reducer
messages   | conversa | _add_messages
usage      | conversa | accumulate_usage
question   | turno    | (sobrescreve)
queries    | turno    | (sobrescreve)
chunks     | turno    | (sobrescreve)
relevant   | turno    | (sobrescreve)
rewrites   | turno    | (sobrescreve)
sources    | turno    | (sobrescreve)
found      | turno    | (sobrescreve)

Recorte do histórico no produto: HISTORY_MESSAGES = 6


Quem escreve cada chave do turno no grafo completo (que montaremos na próxima aula):

| Chave | Escrita por | Para que serve |
|---|---|---|
| `question` | `condense` | pergunta completa, usada na avaliação e na geração |
| `queries` | `condense` (primeira), `rewrite` (acrescenta) | consultas tentadas; a última é a atual |
| `chunks` | `retrieve` | trechos da última busca |
| `relevant` | `grade` | trechos aprovados pela avaliação |
| `rewrites` | `rewrite` | contador para o limite de reescritas |
| `sources`, `found` | `generate`, `no_answer` | fontes citadas e se houve resposta |

Agora o reducer de consumo. Cada nó devolve o consumo de uma chamada (no formato de `token_usage`), e o LangGraph aplica
`accumulate_usage(total_atual, novo)`:

In [10]:
condense_usage = {"input_tokens": 180, "output_tokens": 15, "total_tokens": 195, "calls": 1}
generate_usage = {"input_tokens": 1450, "output_tokens": 90, "total_tokens": 1540, "calls": 1}

total = accumulate_usage(None, {})  # primeiro turno: condense não chamou o modelo
total = accumulate_usage(total, generate_usage)
print("depois do turno 1:", total)
total = accumulate_usage(total, condense_usage)
total = accumulate_usage(total, generate_usage)
print("depois do turno 2:", total)

depois do turno 1: {'input_tokens': 1450, 'output_tokens': 90, 'total_tokens': 1540, 'calls': 1}
depois do turno 2: {'input_tokens': 3080, 'output_tokens': 195, 'total_tokens': 3275, 'calls': 3}


E o schema `RelevanceGrade`, que antecipa a próxima aula. Como no curso de LangChain, o campo de **justificativa vem
primeiro**: o modelo explica antes de decidir quais trechos aprova.

In [11]:
for name, field in RelevanceGrade.model_fields.items():
    print(f"{name}: {field.description}")

example = RelevanceGrade(reasoning="Os trechos 1 e 3 citam o prazo de troca.", relevant=[1, 3])
print("\nexemplo:", example)

reasoning: Frase curta explicando a decisão.
relevant: Números dos trechos úteis. Vazio se nenhum.

exemplo: reasoning='Os trechos 1 e 3 citam o prazo de troca.' relevant=[1, 3]


**Experimente:**

- No grafo da seção 5, troque `HISTORY_MESSAGES` para 2 e rode uma conversa de quatro turnos em que a última pergunta
  depende do primeiro turno. O `condense` ainda resolve a referência?
- Rode a pergunta "O Fone Nova Buds aguenta chuva?" como **segundo** turno de uma conversa sobre outro
  assunto. O `condense` a mantém como está ou a distorce?

## Resumo

- Um RAG vira grafo com dois nós (`retrieve` e `generate`) e um estado desenhado a partir da pergunta "o que cada etapa
  lê e escreve?". Os trechos ficam no estado como `Document`, com os metadados que viram citação.
- Com um checkpointer, o assistente conversa, mas o retriever continua vendo só a string que recebe: a pergunta de
  continuação ("E ele aguenta chuva?") busca outros produtos, e a resposta sai errada ou vazia.
- O nó `condense`, com o `CONDENSE_PROMPT`, reescreve a última mensagem como pergunta completa antes da busca. Custa uma
  chamada por turno (a partir do segundo) e deve ficar registrado em `question`.
- O checkpointer persiste o estado inteiro: chaves do **turno** precisam ser zeradas no primeiro nó da pergunta; chaves
  da **conversa** (`messages`, `usage`) acumulam com reducers.
- No produto, `RAGState`, `accumulate_usage` e `RelevanceGrade` ficam na seção Estado de `src/graph/corrective_rag.py`.

## Próxima aula

Na aula 3.2 o grafo ganha decisões: o nó `grade` avalia se os trechos servem, a aresta condicional escolhe entre gerar,
reescrever a consulta ou desistir com segurança, e montamos o RAG corretivo completo com `build_rag_graph`.